In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("HG=F VS COPX RollingWindow = 30.csv").rename(columns={"index":"date"})
data.head()

,date,HG=F_Close,HG=F_dailyReturns,COPX_Close,COPX_dailyReturns,spread,abs_sprd,HG=F_rolling avg,COPX_rolling avg,isSprdPos,...,HG=F_PnL,COPX_PnL,Total PnL,riskThreshold,grossCashflow,totalMTM,totalMTM in %,isStoppedOut,Adjusted PnL,Gross Capital
0,2010-01-01,3.388,0.000000,32.222782,0.0,0.000000,0.000000,NaN,NaN,1,...,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0,0.0,0.0
1,2010-01-02,3.388,0.000000,32.222782,0.0,0.000000,0.000000,NaN,NaN,1,...,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0,0.0,0.0
2,2010-01-03,3.388,0.000000,32.222782,0.0,0.000000,0.000000,NaN,NaN,1,...,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0,0.0,0.0
3,2010-01-04,3.388,0.000000,32.222782,0.0,0.000000,0.000000,NaN,NaN,1,...,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0,0.0,0.0
4,2010-01-05,3.396,0.002361,32.222782,0.0,0.002361,0.002361,NaN,NaN,1,...,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0,0.0,0.0


In [3]:
# Filter columns to only keep those with "_Close", "_Size", or "_Cashflow"
filtered_columns = [col for col in data.columns if any(keyword in col for keyword in ["date","_Close","_dailyReturns", "_Size", "Live Trades"])]
df = data[filtered_columns]
df.dropna(inplace=True)
df.reset_index(inplace=True, drop=True)
df

C:\Users\tee_m\AppData\Local\Temp\ipykernel_13444\3170661012.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.dropna(inplace=True)


,date,HG=F_Close,HG=F_dailyReturns,COPX_Close,COPX_dailyReturns,HG=F_Size,COPX_Size,Live Trades
0,2010-04-21,3.5330,0.005979,31.717934,-0.015667,-236.0,10.0,Opened
1,2010-04-22,3.4845,-0.013728,31.542332,-0.005536,236.0,-10.0,Closed
2,2010-04-26,3.5270,0.004414,32.134987,0.015257,315.0,-10.0,Opened
3,2010-04-27,3.3635,-0.046357,30.532627,-0.049863,-315.0,10.0,Closed
4,2010-04-30,3.3375,0.001200,29.896076,-0.023656,-1766.0,10.0,Opened
...,...,...,...,...,...,...,...,...
2113,2025-03-10,4.6395,-0.009289,38.610001,-0.033542,-301.0,10.0,Opened
2114,2025-03-11,4.7400,0.021662,39.320000,0.018389,301.0,-10.0,Live
2115,2025-03-12,4.8210,0.017089,39.580002,0.006612,301.0,-10.0,Live
2116,2025-03-13,4.8985,0.016075,40.189999,0.015412,301.0,-10.0,Live


In [4]:
# Extract the part of the column name to the left of "_Close"
close_columns = [col for col in df.columns if "_Close" in col]
left_of_close = [col.split("_Close")[0] for col in close_columns]

# Display the extracted parts
print(left_of_close)
ticker1 = left_of_close[0]
ticker2 = left_of_close[1]

['HG=F', 'COPX']


In [5]:
dir = 0
for i in df.index:
    
    if df.loc[i ,"Live Trades"] == "Opened":
        if df.loc[i, ticker1+"_Size"] > 0:
            dir = 1
        else:
            dir = 0    
        
        while i < len(df) and df.loc[i, "Live Trades"] != "Closed":
            if dir==1:
                df.loc[i, ticker1+"_dir"] = "Long"
                df.loc[i, ticker2+"_dir"] = "Short"
            else:
                df.loc[i, ticker1+"_dir"] = "Short"
                df.loc[i, ticker2+"_dir"] = "Long"   
            i+=1     
            
df.ffill(inplace=True)
df     

C:\Users\tee_m\AppData\Local\Temp\ipykernel_13444\2649518441.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[i, ticker1+"_dir"] = "Short"
C:\Users\tee_m\AppData\Local\Temp\ipykernel_13444\2649518441.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[i, ticker2+"_dir"] = "Long"
C:\Users\tee_m\AppData\Local\Temp\ipykernel_13444\2649518441.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.

,date,HG=F_Close,HG=F_dailyReturns,COPX_Close,COPX_dailyReturns,HG=F_Size,COPX_Size,Live Trades,HG=F_dir,COPX_dir
0,2010-04-21,3.5330,0.005979,31.717934,-0.015667,-236.0,10.0,Opened,Short,Long
1,2010-04-22,3.4845,-0.013728,31.542332,-0.005536,236.0,-10.0,Closed,Short,Long
2,2010-04-26,3.5270,0.004414,32.134987,0.015257,315.0,-10.0,Opened,Long,Short
3,2010-04-27,3.3635,-0.046357,30.532627,-0.049863,-315.0,10.0,Closed,Long,Short
4,2010-04-30,3.3375,0.001200,29.896076,-0.023656,-1766.0,10.0,Opened,Short,Long
...,...,...,...,...,...,...,...,...,...,...
2113,2025-03-10,4.6395,-0.009289,38.610001,-0.033542,-301.0,10.0,Opened,Short,Long
2114,2025-03-11,4.7400,0.021662,39.320000,0.018389,301.0,-10.0,Live,Short,Long
2115,2025-03-12,4.8210,0.017089,39.580002,0.006612,301.0,-10.0,Live,Short,Long
2116,2025-03-13,4.8985,0.016075,40.189999,0.015412,301.0,-10.0,Live,Short,Long


In [6]:
for i in df.index:
    if df.loc[i, "Live Trades"] == "Opened":
        df.loc[i, ticker1+"_MTM"] = abs(df.loc[i, ticker1+"_Size"]*df.loc[i, ticker1+"_Close"])
        df.loc[i, ticker2+"_MTM"] = abs(df.loc[i, ticker2+"_Size"]*df.loc[i, ticker2+"_Close"])
        
        # print(i)
        i+=1
        # print(i)
        while i < len(df) and df.loc[i, "Live Trades"] != "Opened":
            
            if df.loc[i, ticker1+"_dir"] == "Long":
                df.loc[i, ticker1+"_MTM"] = df.loc[i-1, ticker1+"_MTM"] * (1+df.loc[i, ticker1+"_dailyReturns"])
                df.loc[i, ticker2+"_MTM"] = df.loc[i-1, ticker2+"_MTM"] * (1-df.loc[i, ticker2+"_dailyReturns"])

            else:
                df.loc[i, ticker1+"_MTM"] = df.loc[i-1, ticker1+"_MTM"] * (1-df.loc[i, ticker1+"_dailyReturns"])
                df.loc[i, ticker2+"_MTM"] = df.loc[i-1, ticker2+"_MTM"] * (1+df.loc[i, ticker2+"_dailyReturns"])
            
            i+=1
            
        i-=1
    
df

C:\Users\tee_m\AppData\Local\Temp\ipykernel_13444\335835895.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[i, ticker1+"_MTM"] = abs(df.loc[i, ticker1+"_Size"]*df.loc[i, ticker1+"_Close"])
C:\Users\tee_m\AppData\Local\Temp\ipykernel_13444\335835895.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[i, ticker2+"_MTM"] = abs(df.loc[i, ticker2+"_Size"]*df.loc[i, ticker2+"_Close"])


,date,HG=F_Close,HG=F_dailyReturns,COPX_Close,COPX_dailyReturns,HG=F_Size,COPX_Size,Live Trades,HG=F_dir,COPX_dir,HG=F_MTM,COPX_MTM
0,2010-04-21,3.5330,0.005979,31.717934,-0.015667,-236.0,10.0,Opened,Short,Long,833.787998,317.179337
1,2010-04-22,3.4845,-0.013728,31.542332,-0.005536,236.0,-10.0,Closed,Short,Long,845.234013,315.423317
2,2010-04-26,3.5270,0.004414,32.134987,0.015257,315.0,-10.0,Opened,Long,Short,1111.004984,321.349869
3,2010-04-27,3.3635,-0.046357,30.532627,-0.049863,-315.0,10.0,Closed,Long,Short,1059.502537,337.373466
4,2010-04-30,3.3375,0.001200,29.896076,-0.023656,-1766.0,10.0,Opened,Short,Long,5894.025168,298.960762
...,...,...,...,...,...,...,...,...,...,...,...,...
2113,2025-03-10,4.6395,-0.009289,38.610001,-0.033542,-301.0,10.0,Opened,Short,Long,1396.489542,386.100006
2114,2025-03-11,4.7400,0.021662,39.320000,0.018389,301.0,-10.0,Live,Short,Long,1366.239154,393.199997
2115,2025-03-12,4.8210,0.017089,39.580002,0.006612,301.0,-10.0,Live,Short,Long,1342.891933,395.800018
2116,2025-03-13,4.8985,0.016075,40.189999,0.015412,301.0,-10.0,Live,Short,Long,1321.304308,401.899986


In [7]:
df["TotalMTM"] = df[ticker1+"_MTM"] + df[ticker2+"_MTM"]
df["pct_change"] = df["TotalMTM"].pct_change()
df["pct_change"] = np.where(df["Live Trades"] == "Opened", 0 , df["pct_change"])
df["pnl"] = np.where(df["Live Trades"] != "Opened", df["TotalMTM"] - df["TotalMTM"].shift(1), 0)
# df["TotalMTM"].shift(1)
df

C:\Users\tee_m\AppData\Local\Temp\ipykernel_13444\2628036330.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["TotalMTM"] = df[ticker1+"_MTM"] + df[ticker2+"_MTM"]
C:\Users\tee_m\AppData\Local\Temp\ipykernel_13444\2628036330.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["pct_change"] = df["TotalMTM"].pct_change()
C:\Users\tee_m\AppData\Local\Temp\ipykernel_13444\2628036330.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_inde

,date,HG=F_Close,HG=F_dailyReturns,COPX_Close,COPX_dailyReturns,HG=F_Size,COPX_Size,Live Trades,HG=F_dir,COPX_dir,HG=F_MTM,COPX_MTM,TotalMTM,pct_change,pnl
0,2010-04-21,3.5330,0.005979,31.717934,-0.015667,-236.0,10.0,Opened,Short,Long,833.787998,317.179337,1150.967335,0.000000,0.000000
1,2010-04-22,3.4845,-0.013728,31.542332,-0.005536,236.0,-10.0,Closed,Short,Long,845.234013,315.423317,1160.657330,0.008419,9.689995
2,2010-04-26,3.5270,0.004414,32.134987,0.015257,315.0,-10.0,Opened,Long,Short,1111.004984,321.349869,1432.354853,0.000000,0.000000
3,2010-04-27,3.3635,-0.046357,30.532627,-0.049863,-315.0,10.0,Closed,Long,Short,1059.502537,337.373466,1396.876004,-0.024770,-35.478849
4,2010-04-30,3.3375,0.001200,29.896076,-0.023656,-1766.0,10.0,Opened,Short,Long,5894.025168,298.960762,6192.985930,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2113,2025-03-10,4.6395,-0.009289,38.610001,-0.033542,-301.0,10.0,Opened,Short,Long,1396.489542,386.100006,1782.589549,0.000000,0.000000
2114,2025-03-11,4.7400,0.021662,39.320000,0.018389,301.0,-10.0,Live,Short,Long,1366.239154,393.199997,1759.439151,-0.012987,-23.150398
2115,2025-03-12,4.8210,0.017089,39.580002,0.006612,301.0,-10.0,Live,Short,Long,1342.891933,395.800018,1738.691952,-0.011792,-20.747199
2116,2025-03-13,4.8985,0.016075,40.189999,0.015412,301.0,-10.0,Live,Short,Long,1321.304308,401.899986,1723.204294,-0.008908,-15.487658


In [8]:
df["capital"] = np.where(df["Live Trades"] == "Opened", df["TotalMTM"], 0)
total_capital = df["capital"].sum()
df["weighted_avg_capital"] = df["capital"] / total_capital

# Replace 0 with NaN and forward-fill with the previous non-zero value
df["weighted_avg_capital"] = df["weighted_avg_capital"].replace(0, np.nan).ffill()
df["weighted_pct_change"] = df["weighted_avg_capital"] * df["pct_change"]
df

C:\Users\tee_m\AppData\Local\Temp\ipykernel_13444\4069670000.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["capital"] = np.where(df["Live Trades"] == "Opened", df["TotalMTM"], 0)
C:\Users\tee_m\AppData\Local\Temp\ipykernel_13444\4069670000.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["weighted_avg_capital"] = df["capital"] / total_capital
C:\Users\tee_m\AppData\Local\Temp\ipykernel_13444\4069670000.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a Data

,date,HG=F_Close,HG=F_dailyReturns,COPX_Close,COPX_dailyReturns,HG=F_Size,COPX_Size,Live Trades,HG=F_dir,COPX_dir,HG=F_MTM,COPX_MTM,TotalMTM,pct_change,pnl,capital,weighted_avg_capital,weighted_pct_change
0,2010-04-21,3.5330,0.005979,31.717934,-0.015667,-236.0,10.0,Opened,Short,Long,833.787998,317.179337,1150.967335,0.000000,0.000000,1150.967335,0.000723,0.000000
1,2010-04-22,3.4845,-0.013728,31.542332,-0.005536,236.0,-10.0,Closed,Short,Long,845.234013,315.423317,1160.657330,0.008419,9.689995,0.000000,0.000723,0.000006
2,2010-04-26,3.5270,0.004414,32.134987,0.015257,315.0,-10.0,Opened,Long,Short,1111.004984,321.349869,1432.354853,0.000000,0.000000,1432.354853,0.000900,0.000000
3,2010-04-27,3.3635,-0.046357,30.532627,-0.049863,-315.0,10.0,Closed,Long,Short,1059.502537,337.373466,1396.876004,-0.024770,-35.478849,0.000000,0.000900,-0.000022
4,2010-04-30,3.3375,0.001200,29.896076,-0.023656,-1766.0,10.0,Opened,Short,Long,5894.025168,298.960762,6192.985930,0.000000,0.000000,6192.985930,0.003890,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2113,2025-03-10,4.6395,-0.009289,38.610001,-0.033542,-301.0,10.0,Opened,Short,Long,1396.489542,386.100006,1782.589549,0.000000,0.000000,1782.589549,0.001120,0.000000
2114,2025-03-11,4.7400,0.021662,39.320000,0.018389,301.0,-10.0,Live,Short,Long,1366.239154,393.199997,1759.439151,-0.012987,-23.150398,0.000000,0.001120,-0.000015
2115,2025-03-12,4.8210,0.017089,39.580002,0.006612,301.0,-10.0,Live,Short,Long,1342.891933,395.800018,1738.691952,-0.011792,-20.747199,0.000000,0.001120,-0.000013
2116,2025-03-13,4.8985,0.016075,40.189999,0.015412,301.0,-10.0,Live,Short,Long,1321.304308,401.899986,1723.204294,-0.008908,-15.487658,0.000000,0.001120,-0.000010


In [9]:
print(total_capital)
returns = (1+df["weighted_pct_change"]).prod()-1
print(returns)
sd = df["weighted_pct_change"].std()
print(sd)
df["date"] = pd.to_datetime(df["date"])
total_days = df["date"].iloc[-1] - df["date"].iloc[0]
total_days = total_days.days
print(df["pnl"].sum())
returns / (sd*np.sqrt(total_days)) * np.sqrt(252/total_days)

1592111.3538364172
0.005290626899152828
5.640005012633596e-05
8393.238943016193


C:\Users\tee_m\AppData\Local\Temp\ipykernel_13444\821799082.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["date"] = pd.to_datetime(df["date"])


0.27368386236396536

In [10]:
df["weighted_pct_change"].mean() / df["weighted_pct_change"].std() * np.sqrt(252/total_days)

0.009512477791233553